# COMPROBAR DEPENDENCIAS 

In [2]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

print("DuckDB:", duckdb.__version__)
print("Pandas:", pd.__version__)


DuckDB: 1.5.4
Pandas: 3.0.3


# Challenge Meli 

## Estructura del análisis

| Sección | Contenido |
|---|---|
| 1 | Setup e importaciones |
| 2 | Carga y limpieza de datos (Wide → Long) |
| 3 | Carga a DuckDB |
| 4 | Query 1 — Indicadores por Año y País |
| 5 | Query 2 — Crecimiento YoY desde 2010 |
| 6 | Visualizaciones |
| 7 | Conclusiones y relación con MercadoLibre |


# 1. Setup e importaciones 

In [7]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

FILE_CELULAR  = '../Data/BASE_CELULAR.xls'
FILE_INTERNET = '../Data/BASE_INTERNET.xlsx'
FILE_POBLACION = '../Data/BASE_POBLACION.xls'

print('Librerías cargadas')



Librerías cargadas


# 2. Carga y limpieza de datos 

1. Para este paso tenemos que tener en cuenta que los xls menajan los datos en un formato WIDE, en el cual cada año se tiene en una columna. Para trabajar esta data en SQL, necesitamos transformar los datos de WIDE a tipo LONG(País, año, valor).
2. **Agregados regionales:** 49 códigos como `AFE`, `ARB`, `EUU` son sumas regionales, no países. Se excluyen usando la hoja `Metadata - Countries` (campo `Income_Group = 'Agregados'`).
3. **NULLs en 2021:** el 72 % de los países no tiene dato de internet en 2021 porque el World Bank aún no lo había publicado al momento de la extracción. Se conservan como NULL, no se imputan.
4. **Archivo de población (File 3):** contiene datos desde 1960. Solo se usa 2000–2021 para alinear con los otros indicadores.



def leer world bak : Esta función lee un archivo World Bank en formato wide , salta las primeras 3 filas de metadatos del banco mundial y retorna un dataframe limpio.

def wide a long : Con la operación Melt, las columnas de los años la transformamos a una variable que nos dará los años en filas por país.


In [ ]:
def leer_world_bank(path, engine=None):
    kw = {'skiprows': 3, 'header': 0, 'sheet_name': 'Data'}
    if engine:
        kw['engine'] = engine
    df = pd.read_excel(path, **kw)
    return df.dropna(how='all')  


def wide_a_long(df, nombre_valor):
    cols_id   = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']
    cols_anio = [c for c in df.columns if str(c).isdigit()]

    long = df[cols_id + cols_anio].melt(
        id_vars    = cols_id,
        var_name   = 'year',
        value_name = nombre_valor
    )
    long['year'] = long['year'].astype(int)

    return long.rename(columns={
        'Country Name'   : 'country_name',
        'Country Code'   : 'country_code',
        'Indicator Name' : 'indicator_name',
        'Indicator Code' : 'indicator_code',
    })


raw_celular   = leer_world_bank(FILE_CELULAR,   engine='xlrd')
raw_internet  = leer_world_bank(FILE_INTERNET)
raw_poblacion = leer_world_bank(FILE_POBLACION, engine='xlrd')

# Leer metadatos de países (para filtrar agregados y obtener región)
meta = pd.read_excel(FILE_CELULAR, engine='xlrd', sheet_name='Metadata - Countries')
meta = meta.rename(columns={
    'Country Name' : 'country_name',
    'Country Code' : 'country_code',
    'Region'       : 'region',
    'Income_Group' : 'income_group',
})

print(f'File 1 (celular)   : {raw_celular.shape[0]} países, {raw_celular.shape[1]-4} años')
print(f'File 2 (internet)  : {raw_internet.shape[0]} países, {raw_internet.shape[1]-4} años')
print(f'File 3 (población) : {raw_poblacion.shape[0]} países, {raw_poblacion.shape[1]-4} años')
print(f'Metadata           : {len(meta)} registros')


File 1 (celular)   : 266 países, 22 años
File 2 (internet)  : 266 países, 22 años
File 3 (población) : 266 países, 62 años
Metadata           : 266 registros


## Convertir cada xls a long 

Una vez tenemos los formatos Long definidos, transformemos las tablas, y hagamos un Left Join (merge en pandas) desde 2000 a 2021 para unir estas 3 tablas en una sola, sin perder datos en caso tal que falte algun valor en una fila. 

In [10]:
long_celular   = wide_a_long(raw_celular,   'mobile_subscriptions')
long_internet  = wide_a_long(raw_internet,  'internet_pct')
long_poblacion = wide_a_long(raw_poblacion, 'population')


world_indicators = (
    long_internet[['country_name','country_code','year','internet_pct']]
    .merge(long_celular[['country_code','year','mobile_subscriptions']],
           on=['country_code','year'], how='left')
    .merge(long_poblacion[['country_code','year','population']],
           on=['country_code','year'], how='left')
)

# Resumen de calidad
agregados = meta[meta['income_group']=='Agregados']['country_code'].tolist()

print(f'Tabla unificada (world_indicators):')
print(f'  Total filas       : {len(world_indicators):,}')
print(f'  Países totales    : {world_indicators["country_code"].nunique()}')
print(f'  Agregados a excl. : {len(agregados)}')
print(f'  Países reales     : {world_indicators["country_code"].nunique() - len(agregados)}')
print(f'  Rango de años     : {world_indicators["year"].min()} – {world_indicators["year"].max()}')
print()
print('NULLs por indicador:')
print(world_indicators[['internet_pct','mobile_subscriptions','population']].isna().sum())


Tabla unificada (world_indicators):
  Total filas       : 5,852
  Países totales    : 266
  Agregados a excl. : 49
  Países reales     : 217
  Rango de años     : 2000 – 2021

NULLs por indicador:
internet_pct            689
mobile_subscriptions    359
population               22
dtype: int64
